
# Integration Analyst Notebook — Asif

**Role:** Integration Analyst  
**Purpose:** Integrate the cleaned road fatality, population, and weather datasets into one complete monthly state-level dataset from **2017 onward**.




## Integration logic

This notebook performs the following Integration Analyst work:

1. Load datasets provided by previous group members.
2. Use `bitre_state_month` as the main monthly state-level road fatality base.
3. Filter/remove all records before 2017.
4. Build a complete monthly grid for every Australian state/territory from January 2017 to the latest available month.
5. Merge annual population data by `state` and `year`.
6. Merge monthly weather data by `state`, `year`, and `month`.
7. Impute missing values:
   - Missing fatalities are set to 0 only where a state-month row exists in the complete grid but no fatality record is found.
   - Population is forward/back filled by state; the missing 2026 population is filled from the latest available state population.
   - Weather missing values for 2017–2023 are imputed using state-month seasonal medians from available weather data, then state medians, then overall medians.
   - Weather context labels are recalculated from imputed numeric values.
8. Create useful integrated indicators such as fatalities per 100,000 population.
9. Save the final integrated dataset for the next group member.


In [20]:

import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('.')

BITRE_MONTHLY_FILE = 'bitre_state_month.csv'
POPULATION_FILE = 'population_state_year_clean.csv'
WEATHER_FILE = 'weather_monthly_state_clean_2024_jan2026.csv'

START_YEAR = 2017
OUTPUT_FILE = 'asif_integrated_state_month_2017_onward.csv'



## 1. Load cleaned datasets


In [21]:

road_monthly = pd.read_csv(DATA_DIR / BITRE_MONTHLY_FILE, parse_dates=['month_start'])
population = pd.read_csv(DATA_DIR / POPULATION_FILE)
weather = pd.read_csv(DATA_DIR / WEATHER_FILE)

print('Road monthly shape:', road_monthly.shape)
print('Population shape:', population.shape)
print('Weather shape:', weather.shape)

print('\nRoad monthly years:', road_monthly['year'].min(), 'to', road_monthly['year'].max())
print('Population years:', population['year'].min(), 'to', population['year'].max())
print('Weather years:', weather['year'].min(), 'to', weather['year'].max())


Road monthly shape: (3367, 6)
Population shape: (360, 5)
Weather shape: (200, 15)

Road monthly years: 1989 to 2026
Population years: 1981 to 2025
Weather years: 2024 to 2026



## 2. Standardize key fields


In [22]:

# Standardize state codes and date fields
for df in [road_monthly, population, weather]:
    df['state'] = df['state'].astype(str).str.upper().str.strip()

road_monthly['month_start'] = pd.to_datetime(road_monthly['month_start'])
road_monthly['year'] = road_monthly['month_start'].dt.year
road_monthly['month'] = road_monthly['month_start'].dt.month
road_monthly['year_month'] = road_monthly['month_start'].dt.strftime('%Y-%m')

weather['year'] = weather['year'].astype(int)
weather['month'] = weather['month'].astype(int)
weather['year_month'] = pd.to_datetime(weather['year'].astype(str) + '-' + weather['month'].astype(str) + '-01').dt.strftime('%Y-%m')

population['year'] = population['year'].astype(int)

states = sorted(road_monthly['state'].unique())
print('States/territories:', states)


States/territories: ['ACT', 'NSW', 'NT', 'QLD', 'SA', 'TAS', 'VIC', 'WA']



## 3. Filter from 2017 onward and create complete monthly state grid

All records before 2017 are removed because the project focus is from 2017 onward.


In [23]:

road_monthly_2017 = road_monthly[road_monthly['year'] >= START_YEAR].copy()

start_date = pd.Timestamp(f'{START_YEAR}-01-01')
end_date = road_monthly_2017['month_start'].max()
all_months = pd.date_range(start_date, end_date, freq='MS')

complete_grid = pd.MultiIndex.from_product(
    [all_months, states],
    names=['month_start', 'state']
).to_frame(index=False)

complete_grid['year'] = complete_grid['month_start'].dt.year
complete_grid['month'] = complete_grid['month_start'].dt.month
complete_grid['year_month'] = complete_grid['month_start'].dt.strftime('%Y-%m')

print('Complete grid shape:', complete_grid.shape)
print('Expected rows = months x states:', len(all_months), 'x', len(states), '=', len(all_months) * len(states))
print('Date range:', complete_grid['month_start'].min(), 'to', complete_grid['month_start'].max())


Complete grid shape: (872, 5)
Expected rows = months x states: 109 x 8 = 872
Date range: 2017-01-01 00:00:00 to 2026-01-01 00:00:00



## 4. Merge road fatality data


In [24]:

road_for_merge = road_monthly_2017[['month_start', 'state', 'fatalities']].copy()

integrated = complete_grid.merge(
    road_for_merge,
    on=['month_start', 'state'],
    how='left'
)

# In the complete grid, no road row for a state-month means no recorded fatalities in that month.
integrated['fatalities'] = integrated['fatalities'].fillna(0).astype(int)

print('After road merge:', integrated.shape)
print('Missing fatalities:', integrated['fatalities'].isna().sum())


After road merge: (872, 6)
Missing fatalities: 0



## 5. Merge population data

Population is annual by state, so it is merged using `state` and `year`. For 2026, population is filled using the latest available state population because the population file currently ends at 2025.


In [25]:

pop_for_merge = population[['year', 'state', 'state_name', 'population']].copy()

integrated = integrated.merge(
    pop_for_merge,
    on=['year', 'state'],
    how='left'
)

# Fill missing population within each state using nearest available annual value.
integrated = integrated.sort_values(['state', 'month_start'])
integrated['population'] = integrated.groupby('state')['population'].ffill().bfill()
integrated['state_name'] = integrated.groupby('state')['state_name'].ffill().bfill()

integrated['population'] = integrated['population'].round().astype(int)

print('After population merge:', integrated.shape)
print('Missing population:', integrated['population'].isna().sum())


After population merge: (872, 8)
Missing population: 0



## 6. Merge weather data

Weather data is available from Jan 2024 to Jan 2026 only. For 2017–2023, weather is imputed seasonally by state and month.


In [26]:

weather_numeric_cols = [
    'avg_temp_mean',
    'max_temp',
    'min_temp',
    'total_precipitation_mm',
    'total_rain_mm',
    'max_wind_speed_kmh',
    'rainy_days'
]

weather_for_merge = weather[[
    'state', 'city', 'year', 'month', 'year_month',
    *weather_numeric_cols,
    'rain_context', 'heat_context', 'wind_context'
]].copy()

integrated = integrated.merge(
    weather_for_merge,
    on=['state', 'year', 'month', 'year_month'],
    how='left'
)

# Mark whether original weather existed before imputation.
integrated['weather_source'] = np.where(integrated['avg_temp_mean'].notna(), 'Observed weather file', 'Imputed weather')

print('After weather merge:', integrated.shape)
print('Missing weather values before imputation:')
print(integrated[weather_numeric_cols].isna().sum())


After weather merge: (872, 20)
Missing weather values before imputation:
avg_temp_mean             672
max_temp                  672
min_temp                  672
total_precipitation_mm    672
total_rain_mm             672
max_wind_speed_kmh        672
rainy_days                672
dtype: int64



## 7. Impute missing weather values

Imputation method used:

1. `state + month` median, to preserve seasonal pattern for each state.
2. `state` median, if the state-month median is unavailable.
3. Overall median, if anything still remains missing.

This avoids dropping 2017–2023 and keeps the monthly dataset complete.


In [27]:

for col in weather_numeric_cols:
    # Seasonal state-month median
    state_month_median = integrated.groupby(['state', 'month'])[col].transform('median')
    integrated[col] = integrated[col].fillna(state_month_median)

    # State median fallback
    state_median = integrated.groupby('state')[col].transform('median')
    integrated[col] = integrated[col].fillna(state_median)

    # Overall median fallback
    integrated[col] = integrated[col].fillna(integrated[col].median())

# City is a representative weather city per state. Fill by state mode.
def fill_mode_by_group(series):
    modes = series.dropna().mode()
    return modes.iloc[0] if len(modes) else np.nan

state_city_map = weather.groupby('state')['city'].agg(fill_mode_by_group).to_dict()
integrated['city'] = integrated['city'].fillna(integrated['state'].map(state_city_map))

# Round count variable
integrated['rainy_days'] = integrated['rainy_days'].round().astype(int)

print('Missing weather values after imputation:')
print(integrated[weather_numeric_cols + ['city']].isna().sum())


Missing weather values after imputation:
avg_temp_mean             0
max_temp                  0
min_temp                  0
total_precipitation_mm    0
total_rain_mm             0
max_wind_speed_kmh        0
rainy_days                0
city                      0
dtype: int64



## 8. Recalculate weather context labels after imputation

Because many older weather values are imputed, context labels are recalculated consistently from the final numeric weather values.


In [28]:

def classify_rain(mm):
    if mm < 25:
        return 'Low rain'
    elif mm < 75:
        return 'Moderate rain'
    return 'High rain'


def classify_heat(temp):
    if temp < 30:
        return 'Below 30C'
    elif temp <= 35:
        return '30-35C'
    return 'Above 35C'


def classify_wind(kmh):
    if kmh < 40:
        return 'Low wind'
    elif kmh < 70:
        return 'Moderate wind'
    return 'High wind'

integrated['rain_context'] = integrated['total_rain_mm'].apply(classify_rain)
integrated['heat_context'] = integrated['max_temp'].apply(classify_heat)
integrated['wind_context'] = integrated['max_wind_speed_kmh'].apply(classify_wind)

integrated[['rain_context', 'heat_context', 'wind_context']].head()


,rain_context,heat_context,wind_context
0,Moderate rain,30-35C,Low wind
1,Moderate rain,30-35C,Low wind
2,Moderate rain,30-35C,Low wind
3,Moderate rain,Below 30C,Low wind
4,Moderate rain,Below 30C,Low wind



## 9. Add integrated analysis-ready features


In [29]:

integrated['fatalities_per_100k_population'] = (integrated['fatalities'] / integrated['population']) * 100000
integrated['fatalities_per_million_population'] = (integrated['fatalities'] / integrated['population']) * 1000000
integrated['quarter'] = integrated['month_start'].dt.quarter
integrated['month_name'] = integrated['month_start'].dt.month_name()

# Final column order for the next member
final_columns = [
    'month_start', 'year', 'quarter', 'month', 'month_name', 'year_month',
    'state', 'state_name', 'city',
    'fatalities', 'population',
    'fatalities_per_100k_population', 'fatalities_per_million_population',
    'avg_temp_mean', 'max_temp', 'min_temp',
    'total_precipitation_mm', 'total_rain_mm', 'max_wind_speed_kmh', 'rainy_days',
    'rain_context', 'heat_context', 'wind_context', 'weather_source'
]

integrated_final = integrated[final_columns].sort_values(['month_start', 'state']).reset_index(drop=True)
integrated_final.drop(columns=['weather_source'], inplace=True)
print(integrated_final.head())
print('\nFinal shape:', integrated_final.shape)


  month_start  year  quarter  month month_name year_month state  \
0  2017-01-01  2017        1      1    January    2017-01   ACT   
1  2017-01-01  2017        1      1    January    2017-01   NSW   
2  2017-01-01  2017        1      1    January    2017-01    NT   
3  2017-01-01  2017        1      1    January    2017-01   QLD   
4  2017-01-01  2017        1      1    January    2017-01    SA   

                     state_name      city  fatalities  ...  avg_temp_mean  \
0  Australian Capital Territory  Canberra           1  ...      20.109677   
1               New South Wales    Sydney          30  ...      22.293548   
2            Northern Territory    Darwin           3  ...      27.951613   
3                    Queensland  Brisbane          23  ...      24.948387   
4               South Australia  Adelaide           5  ...      23.087097   

   max_temp  min_temp  total_precipitation_mm  total_rain_mm  \
0      33.7       9.6                    49.0           49.0   
1     


## 10. Data quality checks


In [30]:

print('Minimum year:', integrated_final['year'].min())
print('Maximum year:', integrated_final['year'].max())
print('Minimum date:', integrated_final['month_start'].min())
print('Maximum date:', integrated_final['month_start'].max())
print('Rows:', len(integrated_final))
print('States:', integrated_final['state'].nunique())
print('Duplicate state-month rows:', integrated_final.duplicated(['state', 'year_month']).sum())
print('\nMissing values by column:')
print(integrated_final.isna().sum())

expected_rows = len(pd.date_range(integrated_final['month_start'].min(), integrated_final['month_start'].max(), freq='MS')) * integrated_final['state'].nunique()
print('\nExpected complete-grid rows:', expected_rows)
print('Actual rows:', len(integrated_final))

assert integrated_final['year'].min() >= START_YEAR
assert integrated_final.duplicated(['state', 'year_month']).sum() == 0
assert len(integrated_final) == expected_rows
assert integrated_final.isna().sum().sum() == 0
print('\nAll integration checks passed.')


Minimum year: 2017
Maximum year: 2026
Minimum date: 2017-01-01 00:00:00
Maximum date: 2026-01-01 00:00:00
Rows: 872
States: 8
Duplicate state-month rows: 0

Missing values by column:
month_start                          0
year                                 0
quarter                              0
month                                0
month_name                           0
year_month                           0
state                                0
state_name                           0
city                                 0
fatalities                           0
population                           0
fatalities_per_100k_population       0
fatalities_per_million_population    0
avg_temp_mean                        0
max_temp                             0
min_temp                             0
total_precipitation_mm               0
total_rain_mm                        0
max_wind_speed_kmh                   0
rainy_days                           0
rain_context                         

In [31]:
integrated_final

,month_start,year,quarter,month,month_name,year_month,state,state_name,city,fatalities,...,avg_temp_mean,max_temp,min_temp,total_precipitation_mm,total_rain_mm,max_wind_speed_kmh,rainy_days,rain_context,heat_context,wind_context
0,2017-01-01,2017,1,1,January,2017-01,ACT,Australian Capital Territory,Canberra,1,...,20.109677,33.7,9.6,49.0,49.0,27.6,17,Moderate rain,30-35C,Low wind
1,2017-01-01,2017,1,1,January,2017-01,NSW,New South Wales,Sydney,30,...,22.293548,36.8,15.8,153.1,153.1,37.1,23,High rain,Above 35C,Low wind
2,2017-01-01,2017,1,1,January,2017-01,NT,Northern Territory,Darwin,3,...,27.951613,33.2,24.6,288.5,288.5,32.1,30,High rain,30-35C,Low wind
3,2017-01-01,2017,1,1,January,2017-01,QLD,Queensland,Brisbane,23,...,24.948387,37.1,17.5,185.3,185.3,22.5,25,High rain,Above 35C,Low wind
4,2017-01-01,2017,1,1,January,2017-01,SA,South Australia,Adelaide,5,...,23.087097,37.6,10.9,9.3,9.3,31.2,5,Low rain,Above 35C,Low wind
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
867,2026-01-01,2026,1,1,January,2026-01,QLD,Queensland,Brisbane,22,...,24.948387,39.0,17.5,104.8,104.8,22.5,25,High rain,Above 35C,Low wind
868,2026-01-01,2026,1,1,January,2026-01,SA,South Australia,Adelaide,11,...,24.338710,42.6,10.9,2.0,2.0,35.4,3,Low rain,Above 35C,Low wind
869,2026-01-01,2026,1,1,January,2026-01,TAS,Tasmania,Hobart,5,...,16.461290,32.2,8.4,33.5,33.5,30.0,15,Moderate rain,30-35C,Low wind
870,2026-01-01,2026,1,1,January,2026-01,VIC,Victoria,Melbourne,22,...,21.170968,43.3,11.3,12.6,12.6,32.6,13,Low rain,Above 35C,Low wind



## 11. Save final integrated dataset


In [32]:

integrated_final.to_csv(OUTPUT_FILE, index=False)
print(f'Final integrated dataset saved as: {OUTPUT_FILE}')


Final integrated dataset saved as: asif_integrated_state_month_2017_onward.csv



## Final Integration Analyst handoff note

The final file is ready for the next group member. It contains one row per **state-month** from **January 2017 to January 2026**, with road fatalities, annual population, weather variables, imputed missing weather values, and analysis-ready rate indicators.

`data_dictionary.csv` was not merged because it is only an informational metadata file.
